# The 2026 Super El Nino — a climbing index and a warming ocean

By September 2026 this El Nino has become one of the strongest on record. NOAA declared El Nino conditions on **11 June 2026**; the weekly Nino 3.4 index (centred 12 August) reached **+2.7 °C**, and forecasters put a greater-than-90% chance on a *very strong* event through the Northern Hemisphere fall/winter of 2026–27 — with a real chance it exceeds every El Nino on record back to 1950.

This notebook tells that story two ways with the earthlens **`climate-indices`** and **`erddap`** backends: the **Oceanic Nino Index (ONI)** climbing out of a shallow 2024–25 La Nina, and an **animated sea-surface-temperature anomaly** map of the tropical Pacific showing the warm tongue grow month by month through 2026.

> Needs `pyramids-gis[viz]` (cleopatra) for the animation. The ONI series and NOAA Coral Reef Watch SST-anomaly grid are both public — no credentials required.

## Setup

`pyramids` supplies `Dataset` / `DatasetCollection` for reading and animating the SST-anomaly grid; cleopatra supplies the `LineGlyph` line-chart renderer, `apply_blank_canvas` plus the dark reference-map basemap for the animation, and the named `DATA_STYLES` colour presets; `earthlens` supplies the unified `EarthLens` entry point. Every plot in this notebook renders through pyramids or cleopatra — `matplotlib.pyplot` is only used for `plt.show()` / `plt.close()` housekeeping, never for drawing.

In [ ]:
import base64
import warnings
from datetime import datetime
from pathlib import Path
from types import SimpleNamespace

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
from cleopatra.glyphs.primitives.line_glyph import LINE_DEFAULT_OPTIONS, LineGlyph
from cleopatra.styling.colors import DATA_STYLES, resolve_colormap
from cleopatra.styling.styles import apply_blank_canvas
from pyramids.plot import FrameLabel
from IPython.display import HTML
from loguru import logger
from pyramids.dataset import Dataset
from pyramids.dataset.collection import DatasetCollection

from earthlens.core import EarthLens
from earthlens.erddap._helpers import build_constraints, build_griddap_url
from earthlens.erddap.catalog import Catalog

warnings.filterwarnings("ignore")
logger.remove()

OUT = Path("out") / "el_nino_2026"
OUT.mkdir(parents=True, exist_ok=True)

## 1 · The Oceanic Nino Index, 2023–2026

The `climate-indices` backend pulls NOAA PSL's ONI series — a 3-month running mean of the Nino 3.4 SST anomaly. Fetching 2023 onward captures the full prior cycle for context: the 2023–24 El Nino peaking near +2.1 °C, its decay into a shallow 2024–25 La Nina, and this new event's climb.

In [ ]:
oni_dir = OUT / "oni"
oni_dir.mkdir(parents=True, exist_ok=True)

oni_df = EarthLens(
    data_source="climate-indices",
    variables=["oni"],
    start="2023-01-01",
    end="2026-08-31",
    path=oni_dir,
).download(progress_bar=False)

oni = oni_df.dropna(subset=["value"]).assign(date=lambda d: pd.to_datetime(d["date"]))
oni.tail()

### Plot the index

Drawn with cleopatra's `LineGlyph` (`Axes.plot` under the hood, styled through the shared `color_1`/`color_2`/`line_width` options) rather than a bare `matplotlib` call. NOAA's ENSO strength bands (weak/moderate/strong/very strong) are added as reference lines on the glyph's own returned axes — the same "render with the library, adjust via the returned object" pattern used for the raster animation below. The smoothed 3-month ONI lags the raw weekly index — its latest published value understates just how fast this event is moving, so the annotation notes the raw +2.7 °C weekly reading alongside it.

In [ ]:
crimson = LINE_DEFAULT_OPTIONS["color_2"]
line = LineGlyph(
    oni["date"].to_numpy(), oni["value"].to_numpy(), figsize=(10, 4.5), line_width=1.8
)
fig, ax, _ = line.line(color=crimson)

ax.axhline(0, color="0.4", lw=0.8)
for level, band_label in [
    (0.5, "weak"),
    (1.0, "moderate"),
    (1.5, "strong"),
    (2.0, "very strong"),
]:
    ax.axhline(level, color="0.75", lw=0.7, ls="--")
    ax.text(oni["date"].iloc[0], level + 0.04, band_label, fontsize=7, color="0.5")

latest = oni.iloc[-1]
ax.scatter([latest["date"]], [latest["value"]], color=crimson, zorder=5)
ax.annotate(
    f"{latest['value']:+.2f} \u00b0C ({latest['date']:%b %Y}, ONI 3-mo mean)\n"
    "raw weekly Ni\u00f1o 3.4 already +2.7 \u00b0C by mid-Aug 2026",
    xy=(latest["date"], latest["value"]),
    xytext=(-210, 15),
    textcoords="offset points",
    fontsize=8,
    arrowprops=dict(arrowstyle="->", color="0.3"),
)
ax.set(
    ylabel="ONI (\u00b0C)",
    title="Oceanic Ni\u00f1o Index, 2023\u20132026 \u2014 a fast new climb",
)
plt.show()

## 2 · The global SST anomaly, every 10 days from before the event to now

Global coverage, and a window that starts well **before** this El Nino began — January 2025, when the prior shallow La Nina was still near its trough (ONI around −0.45, see the panel above) — through the most recent published day. That covers the full arc from La Nina, through onset, to the current very-strong state. It cannot yet cover the *whole* event: NOAA's forecast has this El Nino still strengthening into winter 2026–27, so its decay hasn't happened yet in the observational record — re-running this cell with a later `end` date once it does extends the animation.

`NOAA_DHW` / `CRW_SSTANOMALY` is a native **5 km** global grid; at full resolution one global day is ~25 million pixels (~200 MB) — cleopatra's animation path builds a full pixel-index list per frame at plot time, so anything much past ~2 million pixels risks a bare `MemoryError`, and 25 million is nowhere close. The `erddap` facade always requests full resolution (no stride kwarg), so this reuses earthlens's own griddap URL builder (`earthlens.erddap._helpers`) directly to ask the **server** to decimate before sending — a `latitude_step`/`longitude_step` of 6 fixes every frame at a constant, safe ~720k pixels regardless of extent, and cuts each day's download by the same 36x. Already-downloaded days are reused, so a rerun does not re-hit the server.

In [ ]:
GLOBAL = {"lat_lim": [-90.0, 90.0], "lon_lim": [-180.0, 180.0]}
STRIDE = 6  # server-side decimation factor -> ~720k px/frame regardless of extent

FRAMES = (
    pd.date_range("2025-01-01", "2026-09-05", freq="10D").strftime("%Y-%m-%d").tolist()
)
if FRAMES[-1] != "2026-09-05":
    FRAMES.append("2026-09-05")  # always include the latest available day

sst_dir = OUT / "sst_anomaly"
sst_dir.mkdir(parents=True, exist_ok=True)

row = Catalog().get("NOAA_DHW")

nc_paths = []
for day in FRAMES:
    nc_path = sst_dir / f"{day}.nc"
    if not nc_path.exists():
        when = datetime.strptime(day, "%Y-%m-%d")
        space = SimpleNamespace(south=-90.0, north=90.0, west=-180.0, east=180.0)
        window = SimpleNamespace(start_date=when, end_date=when)
        constraints = build_constraints(space, window, "griddap")
        constraints["latitude_step"] = STRIDE
        constraints["longitude_step"] = STRIDE
        url = build_griddap_url(
            row.server_url,
            row.dataset_id,
            ["CRW_SSTANOMALY"],
            row.dim_names,
            constraints,
        )
        response = requests.get(url, timeout=120)
        response.raise_for_status()
        nc_path.write_bytes(response.content)
    nc_paths.append(nc_path)

len(nc_paths)

### Mask the fill value and write one GeoTIFF per frame

Reading a NetCDF's variable straight through GDAL's `NETCDF:"...":<var>` subdataset syntax gives back a properly georeferenced `Dataset` — no geotransform to reconstruct by hand. `CRW_SSTANOMALY`'s declared fill value is `-327.68`; anything below `-300` is masked.

In [ ]:
tif_dir = OUT / "sst_anomaly_tif"
tif_dir.mkdir(parents=True, exist_ok=True)

tif_paths = []
for day, nc_path in zip(FRAMES, nc_paths):
    tif_path = tif_dir / f"{day}.tif"
    if not tif_path.exists():
        field = Dataset.read_file(f'NETCDF:"{nc_path}":CRW_SSTANOMALY')
        field = field.apply(lambda v: np.where(v < -300, np.nan, v))
        field.to_file(str(tif_path))
    tif_paths.append(tif_path)

# Short month/year labels -- a full ISO date string gets clipped by the
# animation's frame-label box; "%b %Y" fits cleanly, even repeated across
# the 2-3 frames within the same month.
labels = [pd.to_datetime(day).strftime("%b %Y") for day in FRAMES]
len(tif_paths), labels[0], "->", labels[-1]

## 3 · Animate

cleopatra ships named `DATA_STYLES` presets for dozens of geophysical variables, several of them purpose-built diverging anomaly palettes: `anomaly` (a plain `RdBu_r`), `hot_cold`, `precipitation_anomaly`, `sea_level_anomaly`, and **`temperature_anomaly`** — a dedicated, zero-centred 19-level colour map made specifically for temperature-anomaly fields, not a generic diverging scale reused across variables. That specificity is why it is the pick here over the generic `anomaly`/`hot_cold` presets. Its bounds are widened to ±5 °C to match the source product's own `colorBarMinimum` / `colorBarMaximum` metadata — red where the ocean is warmer than climatology, blue where it is cooler. The warm tongue growing across the central-eastern Pacific through the year is the same warming this notebook's ONI panel already showed as a number.

In [ ]:
west, east = GLOBAL["lon_lim"]
south, north = GLOBAL["lat_lim"]

# `temperature_anomaly`: cleopatra's dedicated diverging preset for temperature-anomaly fields
# (zero-centred, 19 levels) -- purpose-built for this kind of map, unlike the generic `anomaly`
# preset (which is just `RdBu_r`) or `hot_cold`.
anomaly_cmap = resolve_colormap(
    next(iter(DATA_STYLES["temperature_anomaly"].values()))["cmap"]
)

# The global (360x180 deg) box is much wider, relative to the figure, than the
# earlier tropical-Pacific box -- the default frame-label font overflows its
# top-left anchor here, so it is sized down explicitly.
cube = DatasetCollection.from_files(tif_paths)
glyph = cube.plot(
    cmap=anomaly_cmap,
    vmin=-5,
    vmax=5,
    figsize=(9.5, 5.6),
    animation_axis_values=labels,
    frame_label=FrameLabel(color="white", size=10),
)
apply_blank_canvas(glyph.ax, facecolor="black")
glyph.add_reference_map(style="dark", extent=[west, south, east, north])

gif_path = OUT / "el_nino_sst_anomaly_2026.gif"
glyph.save_animation(str(gif_path), fps=5)
plt.close("all")

gif_path

Embed the GIF as a base64 data URI in a plain `<img>` tag — not an HTML5 `<video>` element — so the animation plays in any viewer (Jupyter, VS Code, GitHub, nbviewer, mkdocs) with no JavaScript, matching the pattern the other showcase notebooks in this repo already use.

In [ ]:
encoded = base64.b64encode(gif_path.read_bytes()).decode()
HTML(
    f'<img src="data:image/gif;base64,{encoded}" '
    'alt="Global SST anomaly, Jan 2025 - Sep 2026" />'
)

## Recap

The `climate-indices` ONI series and the `erddap` SST-anomaly grid tell the same story from two angles — a scalar index climbing out of La Nina, and the actual ocean surface warming that drives it — both from public, no-credential earthlens backends.

### Try it yourself

- Narrow the box to the classic Nino 3.4 region (`lon_lim=[-170, -120], lat_lim=[-5, 5]`) for a tighter, more official view.
- Swap in `CRW_DHW` (Degree Heating Weeks) from the same `NOAA_DHW` dataset for the coral-bleaching-stress angle.
- Pair this with the `drought` backend over Indonesia/Australia, or `chc` precipitation over Peru, for the regional-impact half of the story.
- See the [climate-indices](../../reference/climate_indices/introduction.md) and [ERDDAP](../../reference/erddap/introduction.md) backend references.